# Build Wheels for Gaussian Splatting (Fresh Build)

Use this notebook to build `diff-gaussian-rasterization` and `simple-knn` wheels natively on the current Colab environment (CUDA 12.x).
Once built, download the `.whl` files and commit them to the `wheels/` directory in your repository.

In [ ]:
!nvidia-smi
!python --version
!nvcc --version
!pip install ninja

In [ ]:
%cd /content
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting
%cd gaussian-splatting

In [ ]:
%cd /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!pip wheel . --no-deps -w /content/wheels

In [ ]:
%cd /content/gaussian-splatting/submodules/simple-knn
# Note: If simple-knn fails due to missing headers, un-comment the patch below (common issue in CUDA 12 + recent Ubuntu)
# !sed -i '1i#include <float.h>' simple_knn.cu
!pip wheel . --no-deps -w /content/wheels

In [ ]:
!ls -lh /content/wheels

In [ ]:
# Download Widget
import glob
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab import files
import base64

def create_download_widget():
    wheel_files = glob.glob("/content/wheels/*.whl")
    if not wheel_files:
        print("No wheels found in /content/wheels. Did the build succeed?")
        return

    print(f"Found {len(wheel_files)} wheels.")
    
    # Create a zip for convenient downloading
    print("Zipping files...")
    !zip -j -q /content/wheels_bundle.zip /content/wheels/*.whl
    
    btn = widgets.Button(description="Generate Download Link", icon='download')
    output = widgets.Output()

    def on_click(b):
        with output:
            output.clear_output()
            print("Generating link (please wait)...")
            
            # 1. Try automatic download first
            try:
                files.download("/content/wheels_bundle.zip")
            except Exception as e:
                print(f"(Browser download blocked: {e})")
            
            # 2. Always generate a clickable link as fallback/alternative
            try:
                with open("/content/wheels_bundle.zip", "rb") as f:
                    b64 = base64.b64encode(f.read()).decode()
                
                data_uri = f"data:application/zip;base64,{b64}"
                
                # Clickable Link
                href = f'<a href="{data_uri}" download="wheels_bundle.zip" target="_blank" style="font-size: 16px; font-weight: bold; color: #1a0dab; text-decoration: underline; background-color: #f1f3f4; padding: 10px; border-radius: 5px; border: 1px solid #dadce0;">Click here to Download ZIP</a>'
                
                # Copy-Paste Area
                textarea = f'''
                <div style="margin-top: 15px;">
                    <p style="font-size: 12px; color: #666;">If the link/button above fails, copy the Data URI below and paste it into a new tab's address bar:</p>
                    <textarea readonly onclick="this.select();" style="width: 100%; height: 60px; font-size: 10px; color: #555;">{data_uri}</textarea>
                </div>
                '''
                
                display(HTML(href + textarea))
            except Exception as e:
                print(f"Link generation failed: {e}")
            
    btn.on_click(on_click)
    display(widgets.VBox([widgets.Label("Click to generate download options:"), btn, output]))

create_download_widget()

Found 2 wheels.
Zipping files...
